# 04 — Model Experiments

Runs the real training pipeline (`src.models.train.run_training`) and inspects the evaluation report, SHAP attribution, and PSI drift between train/test. Model selection experiments should change `configs/config.yaml` and re-run this, not fork the training code.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from src.config import load_config
from src.models.train import run_training

config = load_config()
metadata = run_training(config)
metadata

In [ ]:
import joblib
import pandas as pd

from src.features.feature_engineering import create_features
from src.models.shap_explainability import plot_summary

isolation_forest = joblib.load(config.paths.models_dir / "isolation_forest.pkl")
sensors = pd.read_parquet(config.paths.sensor_file)
features = create_features(sensors, sensor_columns=config.sensor_columns)
sample = features.dropna(subset=metadata["feature_columns"]).sample(500, random_state=0)
plot_summary(isolation_forest, sample[metadata["feature_columns"]])

In [ ]:
from src.monitoring.drift import compute_feature_drift

train_end = pd.Timestamp(config.split.train_end)
reference = features[features["timestamp"] <= train_end]
current = features[features["timestamp"] > train_end]
drift = compute_feature_drift(reference, current, config.sensor_columns)
drift.feature_psi, drift.warnings, drift.critical